### Phase 1

In [9]:
import numpy as np
import copy

#phase1:setup
#classes, deck setup,turns and states defined

class Card:
    #initializes a card object with a specific color and value
    def __init__(self, color, value):
        self.color = color
        self.value = value
    
    def __repr__(self):
        return f"{self.color} {self.value}"
    #allows direct comparison bw two card objects based on their color and value   
    def __eq__(self, other):
        if isinstance(other, Card):
            return self.color == other.color and self.value == other.value
        return False

def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow'] 
    values = [str(i) for i in range(10)] + ['Skip'] 
    
    deck_list = []
    for c in colors:
        for v in values:
            new_card = Card(c, v)
            deck_list.append(new_card)
            
    deck = np.array(deck_list)
    np.random.shuffle(deck)
    return deck.tolist()

def initialize_game():
    deck = generate_deck()
    state = {
        'p1_cards': [deck.pop() for _ in range(5)], 
        'p2_cards': [deck.pop() for _ in range(5)], 
        'p3_cards': [deck.pop() for _ in range(5)], 
        'top_card': deck.pop(), 
        'deck': deck,
        'current_turn': 1 
    }
    return state

def get_valid_moves(hand, top_card):
    valid_moves = []
    for c in hand:
        if c.color == top_card.color or c.value == top_card.value:
            valid_moves.append(c)
    return valid_moves

def apply_move(state, player_key, move):
    #Deepcopies the state, processes the card play or draw, and advances the turn (handling Skip logic)
    new_state = copy.deepcopy(state)
    
    if move == 'Draw':
        if new_state['deck']:
            new_state[player_key].append(new_state['deck'].pop())
        new_state['current_turn'] = (new_state['current_turn'] % 3) + 1
    else:
        new_state[player_key].remove(move)
        new_state['top_card'] = move
        if move.value == 'Skip':
            new_state['current_turn'] = (new_state['current_turn'] + 1) % 3 + 1
        else:
            new_state['current_turn'] = (new_state['current_turn'] % 3) + 1
            
    return new_state

def is_terminal(state):
    if len(state['p1_cards']) == 0:
        return True
    elif len(state['p2_cards']) == 0:
        return True
    elif len(state['p3_cards']) == 0:
        return True
    else:
        return False

In [10]:
#partial testing of phase 1
test_state = initialize_game()

print("initial states")
print("Top Card:", test_state['top_card'])
print("Player 1 Hand:", test_state['p1_cards'])
print("Player 2 Hand:", test_state['p2_cards'])
print("Player 3 Hand:", test_state['p3_cards'])

#testing valid moves
p1_hand = test_state['p1_cards']
current_top = test_state['top_card']
p1_moves = get_valid_moves(p1_hand, current_top)
print("\nPlayer 1 Valid Moves:", p1_moves)

if p1_moves:
    chosen_move = p1_moves[0]
else:
    chosen_move = 'Draw'

print("Applying Move:", chosen_move)
new_state = apply_move(test_state, 'p1_cards', chosen_move)
print("\nNew Game State")
print("Turn:", new_state['current_turn'])
print("New Top Card:", new_state['top_card'])
print("Player 1 New Hand:", new_state['p1_cards'])

initial states
Top Card: Green 2
Player 1 Hand: [Green 6, Red 5, Yellow 3, Red 6, Red 1]
Player 2 Hand: [Green 1, Yellow 9, Green 9, Blue 9, Red 9]
Player 3 Hand: [Red 7, Green Skip, Yellow 6, Blue 3, Yellow Skip]

Player 1 Valid Moves: [Green 6]
Applying Move: Green 6

New Game State
Turn: 2
New Top Card: Green 6
Player 1 New Hand: [Red 5, Yellow 3, Red 6, Red 1]


### Phase 2

In [11]:
#phase 2: eval function
#adjusts weights etc here
#also drew handwritten part here as well as the explanation of functions in markdown

def get_opponent_keys(player_key):
    players = ['p1_cards', 'p2_cards', 'p3_cards']
    opponents = []
    for p in players:
        if p != player_key:
            opponents.append(p)
    return opponents

def base_evaluation(state, player_key, weights):
    ai_hand = state[player_key]
    opp_keys = get_opponent_keys(player_key)
    c_ai = len(ai_hand)
    
    #calculates average opponent cards
    opp1_key = opp_keys[0]
    opp2_key = opp_keys[1]
    opp1_count = len(state[opp1_key])
    opp2_count = len(state[opp2_key])
    c_opp = (opp1_count + opp2_count) / 2
    s_cards = 0
    for card in ai_hand:
        if card.value == 'Skip':
            s_cards = s_cards + 1

    # score = 50 + (w1 * c_ai) + (w2 * c_opp) + (w3 * s_cards)
    score = 50
    score = score + (weights[0] * c_ai)
    score = score + (weights[1] * c_opp)
    score = score + (weights[2] * s_cards)
    
    return score

def eval_defensive(state, player_key):
    weights = [-5, 2, 3]
    score = base_evaluation(state, player_key, weights)
    return score

def eval_offensive(state, player_key):
    weights = [-8, 1, 1]
    score = base_evaluation(state, player_key, weights)
    return score

In [13]:
#partial testing of phase 2
test_state = initialize_game()

print("hands for eval")
print("Player 1 Hand (Focus AI):", test_state['p1_cards'])
print("Player 2 Hand (Opponent):", test_state['p2_cards'])
print("Player 3 Hand (Opponent):", test_state['p3_cards'])

#counts skips for P1 manually
p1_hand = test_state['p1_cards']
p1_skips = 0
for card in p1_hand:
    if card.value == 'Skip':
        p1_skips = p1_skips + 1

print("\nPlayer 1 currently holds", p1_skips, "'Skip' cards.")
defensive_score = eval_defensive(test_state, 'p1_cards')
offensive_score = eval_offensive(test_state, 'p1_cards')
print("\ncalculated AI Scores for Player 1")
print("Defensive Score [-5, 2, 3]:", defensive_score)
print("Offensive Score [-8, 1, 1]:", offensive_score)

hands for eval
Player 1 Hand (Focus AI): [Yellow 5, Red 7, Yellow 2, Blue 5, Red 3]
Player 2 Hand (Opponent): [Blue Skip, Red 0, Green 1, Green Skip, Blue 8]
Player 3 Hand (Opponent): [Red 8, Green 3, Yellow 0, Red 2, Green 0]

Player 1 currently holds 0 'Skip' cards.

calculated AI Scores for Player 1
Defensive Score [-5, 2, 3]: 35.0
Offensive Score [-8, 1, 1]: 15.0


### Evaluation Functions Explanation

The AI uses a simple formula to score how "good" its current hand is. A higher score means the AI is in a better position to win. 

Formula: Score = 50 - (AI Cards * W1) + (Opponent Cards * W2) + (Skip Cards * W3)

tuned the weights (W) differently for two play styles:

1. Defensive Mode (Weights: -5, 2, 3):** Focuses on survival. It highly values hoarding "Skip" cards to block opponents and prefers opponents to have lots of cards.
2. Offensive Mode (Weights: -8, 1, 1):** Focuses on attacking. It heavily penalizes holding *any* cards, forcing the AI to play its hand aggressively and reach zero cards as fast as possible.